In [ ]:
# 1 Imports
from pathlib import Path

import matplotlib.pyplot as plt

from utils.data_loader import NetatmoDataLoader
from utils.feature_engineer import FeatureEngineer
from utils.split import DatasetSplitter
from utils.metrics import compute_metrics

from models.linear_one_step import LinearOneStepPredictor
from models.ridge_one_step import RidgeOneStepPredictor

In [ ]:
# 2 Load Data
DATA_DIR = Path("../data")

loader = NetatmoDataLoader(DATA_DIR)

loader.station_names()

In [ ]:
df = loader.load_station("Home")

df.head()

In [ ]:
# 3 Feature Engineering
engineer = FeatureEngineer(
    indoor_lags=48,
    outdoor_lags=48,
)

engineer.summary()

In [ ]:
## Transform
df_features = engineer.transform(df)

In [ ]:
## Create dataset
X, y = engineer.create_dataset(df_features)

print(X.shape)
print(y.shape)

In [ ]:
# 4 Split
splitter = DatasetSplitter()

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
) = splitter.split(X, y)

In [ ]:
## Inspect
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# 5 Train Model
model = LinearOneStepPredictor()

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
)

In [ ]:
## Ridge
model = RidgeOneStepPredictor(alpha=1.0)

model.fit(
    X_train,
    y_train,
)

In [ ]:
# 6 Predict
# Validation
y_val_pred = model.predict(X_val)
# Test
y_test_pred = model.predict(X_test)

In [ ]:
# 7 Metrics
# Validation
mae, rmse = compute_metrics(
    y_val,
    y_val_pred,
)

print(f"Validation MAE : {mae:.3f}")
print(f"Validation RMSE: {rmse:.3f}")

# Test
mae, rmse = compute_metrics(
    y_test,
    y_test_pred,
)

print(f"Test MAE : {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")

In [ ]:
# 8 Plot Predictions
plt.figure(figsize=(14,5))

plt.plot(
    y_test.values,
    label="Actual",
)

plt.plot(
    y_test_pred,
    label="Prediction",
)

plt.legend()
plt.title("Indoor temperature prediction")
plt.show()